# Stage 3: Statistical Decomposition & Causal Policy Analysis
This notebook performs statistical analysis to evaluate how monetary policy actions interact with inflation components:
1. **STL Decomposition:** Breaks Core CPI into trend, seasonal, and residual components.
2. **ADF Stationarity Test:** Tests whether inflation series are stationary (a requirement for causality tests).
3. **Granger Causality:** Tests whether movements in Core CPI predict (Granger-cause) subsequent changes in the RBI policy repo rate.
4. **Supply vs. Demand Misattribution Analysis:** Computes the percentage of MPC rate hikes that occurred during supply-shock (food-dominant) inflation environments.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import grangercausalitytests, adfuller
import os

# Load cleaned data
df_cpi = pd.read_csv('../data/cleaned_cpi.csv')
df_cpi['date'] = pd.to_datetime(df_cpi['date'])

df_mpc = pd.read_csv('../data/processed_cpi_mpc.csv')
df_mpc['date'] = pd.to_datetime(df_mpc['date'])

## 1. STL Decomposition of Core CPI YoY
We decompose the Core inflation series to isolate the long-term trend from seasonal fluctuations and noise. Monetary policy is generally designed to react to the persistent trend.

In [ ]:
core_series = df_cpi.set_index('date')['cpi_core_yoy'].dropna()

# Fit Seasonal-Trend decomposition using Loess (STL) with period=12 (monthly)
stl = STL(core_series, period=12, robust=True)
result = stl.fit()

# Plot the STL components
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
axes[0].plot(core_series, color='#2c3e50', linewidth=1.8)
axes[0].set_ylabel('Observed (YoY %)')
axes[0].set_title('STL Decomposition of India Core CPI Inflation', fontsize=12)

axes[1].plot(result.trend, color='#e74c3c', linewidth=1.8)
axes[1].set_ylabel('Trend (YoY %)')

axes[2].plot(result.seasonal, color='#27ae60', linewidth=1.8)
axes[2].set_ylabel('Seasonal (YoY %)')

axes[3].plot(result.resid, color='#95a5a6', linewidth=1.5)
axes[3].set_ylabel('Residual (YoY %)')
axes[3].axhline(0, color='black', linewidth=0.8, linestyle=':')

for ax in axes:
    ax.grid(True, alpha=0.2)
    
plt.tight_layout()
os.makedirs('../outputs', exist_ok=True)
plt.savefig('../outputs/03_stl_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

# Save the extracted trend back to the main CPI dataset
df_cpi['core_trend'] = np.nan
df_cpi.loc[df_cpi['date'].isin(core_series.index), 'core_trend'] = result.trend.values
df_cpi.to_csv('../data/cleaned_cpi.csv', index=False)
print("STL Decomposition complete, core_trend saved back to data/cleaned_cpi.csv.")

## 2. Stationarity Testing (ADF Test)
Before testing for Granger causality, we run the Augmented Dickey-Fuller (ADF) test to verify if the inflation series are stationary.

In [ ]:
print("=== Augmented Dickey-Fuller (ADF) Stationarity Test ===")
adf_core = adfuller(df_cpi['cpi_core_yoy'].dropna())
adf_general = adfuller(df_cpi['cpi_general_yoy'].dropna())

print(f"Core CPI ADF statistic: {adf_core[0]:.4f}, p-value: {adf_core[1]:.4f} "
      f"({'stationary' if adf_core[1] < 0.05 else 'non-stationary'})")
print(f"General CPI ADF statistic: {adf_general[0]:.4f}, p-value: {adf_general[1]:.4f} "
      f"({'stationary' if adf_general[1] < 0.05 else 'non-stationary'})")

## 3. Granger Causality: Does Core CPI YoY predict Repo Rate Changes?
To run the causality test monthly, we merge the policy repo rate (which changes at meeting dates) back onto the monthly CPI dataset using a backward merge.

In [ ]:
# Merge monthly repo rate into CPI
cpi_mpc = pd.merge_asof(
    df_cpi.sort_values('date'),
    df_mpc[['date', 'repo_rate']].sort_values('date'),
    on='date',
    direction='backward'
).dropna(subset=['cpi_core_yoy', 'repo_rate'])

print(f"Merged monthly CPI-MPC dataset shape: {cpi_mpc.shape}")

# Run Granger Causality Test
# Key question: does core inflation Granger-cause repo rate movements?
# Series array format for statsmodels: [target_variable, predictor_variable]
gc_data = cpi_mpc[['repo_rate', 'cpi_core_yoy']].dropna()
print("\n=== Granger Causality Test Results (Core CPI -> Repo Rate) ===")
gc_results = grangercausalitytests(gc_data, maxlag=6, verbose=True)

### Policy Interpretation:
* If the p-value is **significant (< 0.05)** for lags 1 or 2, it suggests that past core inflation trends reliably predict monetary policy actions (repo rate changes). This indicates a responsive monetary policy that acts on demand-side signals.

## 4. Supply vs. Demand Misattribution Analysis
We calculate the proportion of rate hikes that occurred when food inflation (supply-side shock) was higher than core inflation (demand-side inflation).

In [ ]:
# Filter for rate hike decisions
df_hikes = df_mpc[df_mpc['decision'] == 'hike'].copy()
df_hikes['food_dominant'] = df_hikes['cpi_food_yoy'] > df_hikes['cpi_core_yoy']

total_hikes = len(df_hikes)
supply_hikes = df_hikes[df_hikes['food_dominant'] == True]
supply_hikes_count = len(supply_hikes)
percentage = (supply_hikes_count / total_hikes * 100) if total_hikes > 0 else 0

print("=== Supply vs. Demand Misattribution Analysis ===")
print(f"Total rate hikes executed since Oct 2016: {total_hikes}")
print(f"Hikes executed during food-dominated inflation environments: {supply_hikes_count} ({percentage:.1f}%)")

if supply_hikes_count > 0:
    print("\nHiking episodes during supply-shock dominance:")
    print(supply_hikes[['date', 'repo_rate', 'bps_change', 'cpi_general_yoy', 'cpi_food_yoy', 'cpi_core_yoy']].to_string(index=False))

### Policy Interpretation:
* Since rate hikes are intended to cool aggregate demand (which affects Core CPI), hiking when food inflation dominates (supply shocks) introduces the risk of economic growth costs without cooling the structural drivers of inflation. Finding that a substantial percentage of hikes occurred in food-dominant environments suggests the MPC might have felt compelled to react to headline inflation spikes, risking over-tightening.